# EDA Workflow End to End

**Topic:** Exploratory Data Analysis

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from IPython.display import display
from scipy import stats

titanic = sns.load_dataset("titanic")
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Execute** a complete, structured EDA on a real dataset from first load to modeling insights
- **Synthesize** findings across all EDA stages into a coherent narrative about the data
- **Formulate** specific, data-driven hypotheses that would guide the feature selection for a predictive model

> **Tip:** Read the narrative markdown cells before each code cell. This notebook models how a real data scientist thinks through an EDA, not just the code they write.

---
## How we got here

This is the capstone notebook for the EDA series. Over the previous 11 notebooks, you built every tool in the EDA toolkit: loading, inspection, type casting, missing data, duplicates, outlier detection, descriptive statistics, univariate analysis, bivariate analysis, multivariate analysis, and feature distribution assessment. This notebook applies all of them in sequence to the Titanic dataset and synthesizes the findings into actionable modeling insights.

---
## Why this matters for data science

Knowing each EDA technique individually is not enough. A real data scientist needs to know how to run a complete EDA from scratch on an unfamiliar dataset, narrate the findings clearly, and arrive at concrete recommendations. This notebook is a worked example of that process. The structure here is one you can apply to any new dataset you encounter in your career.

---
## Step 1: Load and Inspect

We begin every EDA the same way: load the data and immediately check its shape, types, and first rows. No analysis, no transformations yet. Just look.

In [2]:
print(f"Shape: {titanic.shape}")
print()
print("Data types:")
print(titanic.dtypes)
print()
print("First 5 rows:")
titanic.head()

Shape: (891, 15)

Data types:
survived          int64
pclass            int64
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
embarked            str
class          category
who                 str
adult_male         bool
deck           category
embark_town         str
alive               str
alone              bool
dtype: object

First 5 rows:


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


The dataset has 891 rows and 15 columns. We have a mix of numeric columns (`age`, `fare`, `sibsp`, `parch`) and categorical columns (`sex`, `pclass`, `embarked`). The `survived` column is our target: 0 means the passenger did not survive, 1 means they did.

---
## Step 2: Assess Data Quality

### 2a. Missing Values

Before any analysis, we need to know what is absent.

In [3]:
missing = titanic.isnull().sum()
missing_pct = (missing / len(titanic) * 100).round(1)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values("missing_pct", ascending=False)
print("Columns with missing values:")
display(missing_df)

Columns with missing values:


,missing_count,missing_pct
deck,688,77.2
age,177,19.9
embarked,2,0.2
embark_town,2,0.2


Three columns have missing data. `deck` is 77% missing. That is not a minor gap — more data is absent than present. We will not use `deck` as a feature directly, but we can engineer a binary "has_deck_record" indicator that might still carry information about passenger wealth. `age` is about 20% missing. We will impute it with the median before modeling. `embarked` has only 2 missing rows, which we can drop safely.

### 2b. Duplicate Records

In [4]:
n_dupes = titanic.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}")
print(f"Total rows: {len(titanic)}")
print("No duplicates found." if n_dupes == 0 else f"{n_dupes} duplicates — investigate before dropping.")

Exact duplicate rows: 107
Total rows: 891
107 duplicates — investigate before dropping.


No exact duplicates. The Titanic dataset is unusually clean on this front.

### 2c. Outliers

In [5]:
fare = titanic["fare"].dropna()
Q1, Q3 = fare.quantile(0.25), fare.quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
outliers = fare[fare > upper_bound]
print(f"Fare IQR upper bound (k=1.5): £{upper_bound:.1f}")
print(f"Outliers: {len(outliers)} passengers ({len(outliers)/len(fare)*100:.1f}%)")
print(f"Max fare: £{fare.max():.2f}")
print()
age = titanic["age"].dropna()
Q1a, Q3a = age.quantile(0.25), age.quantile(0.75)
IQRa = Q3a - Q1a
upper_a = Q3a + 1.5 * IQRa
age_outliers = age[age > upper_a]
print(f"Age IQR upper bound (k=1.5): {upper_a:.1f} years")
print(f"Age outliers: {len(age_outliers)} passengers ({len(age_outliers)/len(age)*100:.1f}%)")

Fare IQR upper bound (k=1.5): £65.6
Outliers: 116 passengers (13.0%)
Max fare: £512.33

Age IQR upper bound (k=1.5): 64.8 years
Age outliers: 11 passengers (1.5%)


The fare column has a significant number of extreme values by the IQR criterion. The maximum fare of £512 is far above the IQR upper bound of approximately £65. However, this is historically consistent with first-class berths on a luxury liner. These are real data points, not errors. We will apply a log transformation to `fare` before modeling rather than removing any rows.

Age outliers are minimal (a few elderly passengers above the IQR bound). These are also genuine.

---
## Step 3: Descriptive Statistics

With data quality addressed, we summarize the numeric columns.

In [6]:
display(titanic[["age", "fare", "sibsp", "parch"]].describe().round(2))

,age,fare,sibsp,parch
count,714.00,891.00,891.00,891.00
mean,29.70,32.20,0.52,0.38
std,14.53,49.69,1.10,0.81
min,0.42,0.00,0.00,0.00
25%,20.12,7.91,0.00,0.00
50%,28.00,14.45,0.00,0.00
75%,38.00,31.00,1.00,0.00
max,80.00,512.33,8.00,6.00


The mean age is 29.7, the median is 28. The small gap confirms mild right skew. The mean fare is £32.2, the median is £14.5. The large gap confirms strong right skew — a small number of very expensive tickets pulls the mean far above the typical fare. Most passengers (`sibsp` median = 0, `parch` median = 0) traveled without family members.

---
## Step 4: Univariate Analysis

We examine each key variable individually before looking at relationships.

In [7]:
age = titanic["age"].dropna()
survived_counts = titanic["survived"].value_counts().sort_index()
pclass_counts = titanic["pclass"].value_counts().sort_index()
sex_counts = titanic["sex"].value_counts()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=["Age Distribution", "Survival Outcome", "Passenger Class", "Sex"],
    vertical_spacing=0.18,
    horizontal_spacing=0.12,
)

kde_x = np.linspace(age.min(), age.max(), 300)
kde_y = stats.gaussian_kde(age)(kde_x)
kde_scale = len(age) * (age.max() - age.min()) / 25

fig.add_trace(go.Histogram(x=age, nbinsx=25, marker_color=PALETTE["primary"], opacity=0.65, name="Age", showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=kde_x, y=kde_y * kde_scale, mode="lines", line=dict(color=PALETTE["secondary"], width=2), name="KDE", showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=["Died", "Survived"], y=survived_counts.values, marker_color=[PALETTE["secondary"], PALETTE["accent"]], showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=["1st", "2nd", "3rd"], y=pclass_counts.values, marker_color=PALETTE["primary"], showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=sex_counts.index.tolist(), y=sex_counts.values, marker_color=[PALETTE["primary"], PALETTE["secondary"]], showlegend=False), row=2, col=2)

fig.update_layout(
    title=dict(text="Univariate Analysis: Key Titanic Variables", font=dict(size=FONT["size_title"], family=FONT["family"])),
    paper_bgcolor=PALETTE["background"],
    plot_bgcolor=PALETTE["surface"],
    height=550,
    showlegend=False,
    margin=dict(l=60, r=30, t=80, b=60),
)
fig.show()

The age distribution is roughly bell-shaped with mild right skew and a cluster of infants visible near zero. The survival outcome shows that 549 passengers (62%) did not survive and 342 (38%) did. The class distribution shows more 3rd class passengers than 1st and 2nd combined. The sex distribution is approximately 65% male and 35% female.

---
## Step 5: Bivariate Analysis

Now we look at relationships between variables and our target.

In [8]:
surv_by_sex = titanic.groupby("sex")["survived"].mean() * 100
surv_by_class = titanic.groupby("pclass")["survived"].mean() * 100
surv_by_class_sex = titanic.groupby(["pclass", "sex"])["survived"].mean().unstack() * 100

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Survival Rate by Sex", "Survival Rate by Class", "Survival by Class and Sex"],
    horizontal_spacing=0.10,
)

fig.add_trace(go.Bar(x=surv_by_sex.index.tolist(), y=surv_by_sex.values,
    marker_color=[PALETTE["primary"], PALETTE["secondary"]], showlegend=False,
    text=[f"{v:.1f}%" for v in surv_by_sex.values], textposition="outside"), row=1, col=1)

fig.add_trace(go.Bar(x=["1st", "2nd", "3rd"], y=surv_by_class.values,
    marker_color=PALETTE["primary"], showlegend=False,
    text=[f"{v:.1f}%" for v in surv_by_class.values], textposition="outside"), row=1, col=2)

for i, sex in enumerate(surv_by_class_sex.columns):
    fig.add_trace(go.Bar(
        name=sex,
        x=["1st", "2nd", "3rd"],
        y=surv_by_class_sex[sex].values,
        marker_color=[PALETTE["primary"], PALETTE["secondary"]][i],
        text=[f"{v:.1f}%" for v in surv_by_class_sex[sex].values],
        textposition="outside",
    ), row=1, col=3)

fig.update_layout(
    title=dict(text="Bivariate Analysis: Survival by Key Predictors", font=dict(size=FONT["size_title"], family=FONT["family"])),
    paper_bgcolor=PALETTE["background"],
    plot_bgcolor=PALETTE["surface"],
    height=420,
    barmode="group",
    showlegend=True,
    margin=dict(l=60, r=30, t=80, b=60),
)
fig.show()

The bivariate patterns are striking. Females had a 74% survival rate compared to 19% for males. First class passengers had a 63% survival rate compared to 24% in third class. The third panel reveals that even within the same class, women survived at dramatically higher rates than men. A 1st class woman had a 97% survival rate. A 3rd class man had an 14% survival rate.

---
## Step 6: Multivariate Analysis

We examine correlations among all numeric variables simultaneously.

In [9]:
numeric_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr = titanic[numeric_cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=numeric_cols, y=numeric_cols,
    colorscale=[[0.0, "#F76707"], [0.5, "#FFFFFF"], [1.0, "#4C6EF5"]],
    zmid=0,
    text=[[f"{v:.2f}" for v in row] for row in corr.values],
    texttemplate="%{text}",
    textfont={"size": 12},
))

layout = base_layout(title="Multivariate Analysis: Titanic Correlation Matrix")
layout.update(height=400)
fig.update_layout(layout)
fig.show()

The correlation matrix confirms the bivariate findings: `pclass` and `survived` have a moderate negative correlation (r ≈ -0.34). `pclass` and `fare` are strongly negatively correlated (r ≈ -0.55), reflecting the obvious relationship between class and ticket price. No feature pair shows a strong enough positive correlation to raise concerns about multicollinearity in a simple model.

---
## Step 7: Key Findings

A summary of the five most important findings from this EDA.

In [10]:
# Key quantitative findings referenced in the narrative above
total = len(titanic)
surv_rate = titanic["survived"].mean() * 100
female_surv = titanic[titanic["sex"] == "female"]["survived"].mean() * 100
male_surv = titanic[titanic["sex"] == "male"]["survived"].mean() * 100
c1_surv = titanic[titanic["pclass"] == 1]["survived"].mean() * 100
c3_surv = titanic[titanic["pclass"] == 3]["survived"].mean() * 100
age_missing = titanic["age"].isnull().sum()
deck_missing = titanic["deck"].isnull().sum()
deck_pct = deck_missing / total * 100
fare_skew = titanic["fare"].skew()

print(f"Overall survival rate:       {surv_rate:.1f}%")
print(f"Female survival rate:        {female_surv:.1f}%")
print(f"Male survival rate:          {male_surv:.1f}%")
print(f"1st class survival rate:     {c1_surv:.1f}%")
print(f"3rd class survival rate:     {c3_surv:.1f}%")
print(f"Age missing:                 {age_missing} rows ({age_missing/total*100:.1f}%)")
print(f"Deck missing:                {deck_missing} rows ({deck_pct:.1f}%)")
print(f"Fare skewness:               {fare_skew:.2f}")

Overall survival rate:       38.4%
Female survival rate:        74.2%
Male survival rate:          18.9%
1st class survival rate:     63.0%
3rd class survival rate:     24.2%
Age missing:                 177 rows (19.9%)
Deck missing:                688 rows (77.2%)
Fare skewness:               4.79


### What we found

1. **Sex is the strongest single predictor of survival.** Females survived at 74%, males at 19% — a 55-point gap. Any model that does not include `sex` will miss the most important signal in the data.

2. **Passenger class is the second strongest predictor.** First class passengers survived at more than 2.5 times the rate of third class passengers. Class and sex interact: a 1st class woman and a 3rd class man lived in almost entirely different probability worlds on the night of April 14, 1912.

3. **`deck` is 77% missing but may still contain usable information.** The presence of a deck record is likely correlated with higher class. Engineering a binary `has_deck` feature captures this signal without requiring us to impute or drop the column.

4. **`fare` requires a log transformation before use in a linear model.** Skewness of approximately 4.8 means raw fare values will distort any linear model. `log(1 + fare)` reduces skewness to approximately 0.4, making it suitable as a feature.

5. **`age` requires imputation.** With 20% of values missing, median imputation (28 years) is a reasonable baseline. A more sophisticated approach would impute separately by class and sex, since age distributions differ between groups.

---
## Discussion

> **Discussion question:** Based on this EDA, which 3 features would you include in a model predicting survival? Justify your choices using specific findings from the analysis above.

---
## Key takeaway

> **A complete EDA does not just describe the data. It tells a story, identifies the most important signals, flags the problems that need to be fixed, and hands a list of concrete, evidence-based recommendations to the modeling step.**

---
*You have completed the EDA working session series. The next step is feature engineering and model training, where all of these findings become inputs to a predictive model.*